# Phase 2 specialist track — CheXagent-2 naming (Colab)

Runs the **medical-specialist VLM CheXagent-2 (3B)** on the *same* V4b tube crops the general
VLMs saw, under the *same blind prompt*, writing `predictions_chexagent.jsonl` in the shared
file contract. Download it and score locally with:
`python -m auditor.discovery score --manifest manifest.json --predictions predictions_chexagent.jsonl --confound tube`

**Runtime -> Change runtime type -> T4 GPU** before running.
**Upload** `crops.zip` (the V4b tube crops) and `manifest.json`.

*Note:* CheXagent's exact processor / `generate` API may differ from the snippet below — verify
against the current model card at `StanfordAIMI/CheXagent-2-3b` at run time; this cell uses the
documented interface. If the API differs, adjust the load/generate cell only — the file-contract
output format (one `{"id", "description"}` JSON line per crop) must stay identical for scoring.

In [ ]:
# 1. Dependencies
!pip -q install transformers accelerate pillow

In [ ]:
# 2. Upload crops.zip + manifest.json
from google.colab import files
import zipfile, os
print('Upload crops.zip and manifest.json ...')
up = files.upload()
if 'crops.zip' in up:
    with zipfile.ZipFile('crops.zip') as z:
        z.extractall('.')
assert os.path.isdir('crops'), 'crops/ folder not found after unzip'
assert os.path.exists('manifest.json'), 'manifest.json not found'
print('crops:', len(os.listdir('crops')), '| manifest present')

In [ ]:
# 3. Load CheXagent-2 (3B) in fp16 on the T4
import torch
from transformers import AutoModelForCausalLM, AutoProcessor
MODEL = 'StanfordAIMI/CheXagent-2-3b'
processor = AutoProcessor.from_pretrained(MODEL, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL, torch_dtype=torch.float16, trust_remote_code=True, device_map='auto').eval()
print('model loaded')

In [ ]:
# 4. Blind-prompt loop -> predictions_chexagent.jsonl (same prompt/contract as the general VLMs)
import json
from PIL import Image
PROMPT = ('This is a small region a diagnostic model focused on in a chest X-ray. '
          'Describe anything artificial, non-anatomical, or out of place. If it looks normal, say so.')
manifest = json.load(open('manifest.json'))
with open('predictions_chexagent.jsonl', 'w') as f:
    for i, row in enumerate(manifest):
        cid = row['id']
        img = Image.open(f'crops/{cid}.png').convert('RGB')
        inputs = processor(images=img, text=PROMPT, return_tensors='pt').to(model.device)
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=120, do_sample=False)
        desc = processor.batch_decode(out, skip_special_tokens=True)[0].strip()
        f.write(json.dumps({'id': cid, 'description': desc}) + '\n')
        if i % 20 == 0:
            print(i, cid, desc[:70])
print('DONE ->', len(manifest), 'predictions')

In [ ]:
# 5. Download for local scoring
from google.colab import files
files.download('predictions_chexagent.jsonl')